In [ ]:
import math
import torchvision.transforms as transforms
from torchvision.utils import save_image
import PIL as pil
import os
import cv2
import torchvision.transforms.functional as F
import numpy as np
import torchvision
import torch
from tqdm import tqdm
import random
from albumentations import (
    Compose,
    HorizontalFlip,
    VerticalFlip,
    Rotate,
    RandomBrightnessContrast,
    HueSaturationValue,
    GaussianBlur,
    ShiftScaleRotate,
    Resize,
)

In [ ]:
IMG_SIZE = 256
IMG_DIR = "data/"
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count() - 2

In [ ]:
class BenGrahamPreprocessing:
    def __init__(self, img_size):
        self.img_size = img_size

    def __call__(self, image: pil.Image.Image) -> pil.Image.Image:
        image_np = np.array(image)

        # Aplicar el desenfoque gaussiano de OpenCV
        blurred_image = cv2.GaussianBlur(image_np, (0, 0), self.img_size / 10)

        # Combinar la imagen original y la desenfocada
        combined_image = cv2.addWeighted(image_np, 4, blurred_image, -4, 128)

        # Convertir a escala de grises
        return F.to_pil_image(combined_image)

In [ ]:
class BackgroundRemover:
    def __init__(self, version=1):
        self.version = version
        pass

    def __call__(self, image: pil.Image.Image) -> pil.Image.Image:
        self.ancho, self.alto = image.size
        if self.version == 1:
            return self.recortar_bordes_negros(image)
        else:
            return self.recortar_bordes_negros_v2(image)

    def encontrar_limites(self, img):
        pixeles = img.load()

        ancho, alto = self.ancho, self.alto
        ancho_medio = ancho // 2
        alto_medio = alto // 2
        x_inicio = 0
        x_inicio = 0
        for x in range(ancho):
            if pixeles[x, alto_medio] != (0, 0, 0):
                x_inicio = x
                break

        x_fin = ancho - 1
        for x in range(ancho - 1, -1, -1):
            if pixeles[x, alto_medio] != (0, 0, 0):
                x_fin = x
                break

        y_inicio = 0
        for y in range(alto):
            if pixeles[ancho_medio, y] != (0, 0, 0):
                y_inicio = y
                break

        y_fin = alto - 1
        for y in range(alto - 1, -1, -1):
            if pixeles[ancho_medio, y] != (0, 0, 0):
                y_fin = y
                break

        return x_inicio, x_fin, y_inicio, y_fin

    def calcular_centro_y_radio(self, x_inicio, x_fin, y_inicio, y_fin):
        centro_x = (x_inicio + x_fin) // 2
        centro_y = (y_inicio + y_fin) // 2
        radio = max((x_fin - x_inicio) // 2, (y_fin - y_inicio) // 2)
        return centro_x, centro_y, radio

    def recortar_ojo(self, imagen, centro_x, centro_y, radio):
        lado = int((2 * radio) / math.sqrt(2))
        mitad_lado = lado // 2

        izquierda = max(centro_x - mitad_lado, 0)
        arriba = max(centro_y - mitad_lado, 0)
        derecha = min(centro_x + mitad_lado, imagen.width)
        abajo = min(centro_y + mitad_lado, imagen.height)

        imagen_recortada = imagen.crop((izquierda, arriba, derecha, abajo))
        return imagen_recortada

    def recortar_bordes_negros(self, imagen):
        x_inicio, x_fin, y_inicio, y_fin = self.encontrar_limites(imagen)

        imagen_recortada = imagen.crop((x_inicio, y_inicio, x_fin + 1, y_fin + 1))
        return imagen_recortada

    def recortar_bordes_negros_v2(self, img):
        x_inicio, x_fin, y_inicio, y_fin = self.encontrar_limites(img)

        centro_x, centro_y, radio = self.calcular_centro_y_radio(
            x_inicio, x_fin, y_inicio, y_fin
        )

        return self.recortar_ojo(img, centro_x, centro_y, radio)

In [ ]:
def define_transform(imgsize, bg_rem_version, custom_prep):
    tranforms = [
        BackgroundRemover(version=bg_rem_version),
        transforms.Resize((imgsize, imgsize)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.ToTensor(),
        # transforms.Normalize(mean = [0.485, 0.456, 0.406],std = [0.229, 0.224, 0.225]),
    ]
    if custom_prep:
        tranforms.insert(5, BenGrahamPreprocessing(IMG_SIZE))

    return transforms.Compose(tranforms)

In [ ]:
def define_loaders(transform):
    trainset = torchvision.datasets.ImageFolder(root="data/train/", transform=transform)
    trainloader = torch.utils.data.DataLoader(
        trainset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    testset = torchvision.datasets.ImageFolder(root="data/test/", transform=transform)
    testloader = torch.utils.data.DataLoader(
        testset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    return trainloader, testloader

In [ ]:
def save_preprocessed_images(
    loader, loader_name, img_size, bg_rem_version, custom_prep
):
    info_str = f"S_{img_size}_R_{bg_rem_version}_C_{custom_prep}"
    # Create output folder with loader name
    output_folder = f"data/prep/{info_str}/{loader_name}"
    os.makedirs(output_folder, exist_ok=True)

    # Create subfolders for each class
    for class_idx in range(len(loader.dataset.classes)):
        os.makedirs(os.path.join(output_folder, str(class_idx)), exist_ok=True)

    # Save transformed images
    for i, (image, label) in tqdm(
        enumerate(loader), desc=f"Processing {loader_name}", total=len(loader)
    ):
        for j in range(image.size(0)):
            # Get class folder for current image
            class_folder = str(label[j].item())
            # Save image in corresponding class folder
            save_image(
                image[j],
                os.path.join(
                    output_folder, class_folder, f"image_{i * BATCH_SIZE + j}.png"
                ),
            )

    print(
        f"Transformed images saved to '{output_folder}' in class-specific subfolders."
    )

In [ ]:
from itertools import product

# Define configurations
configs = {
    "bg_rem_version": [1, 2],
    "custom_prep": [False],
    "image_size": [224],  # , 256, 299]
}

# Process all combinations
for bg_rem_version, custom_prep, image_size in product(*configs.values()):
    transform = define_transform(image_size, bg_rem_version, custom_prep)
    trainloader, testloader = define_loaders(transform)

    # Process both train and test sets
    for loader, split in [(trainloader, "train"), (testloader, "test")]:
        save_preprocessed_images(loader, split, image_size, bg_rem_version, custom_prep)

In [ ]:
def augment_and_balance_dataset(dataset_path, augmentation_pipeline):
    image_paths = []
    labels = []
    class_names = sorted(os.listdir(dataset_path))  # Sort class names consistently
    biggest_class_size = 0
    for class_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(dataset_path, class_name)
        if os.path.isdir(class_dir):
            for image_name in os.listdir(class_dir):
                image_path = os.path.join(class_dir, image_name)
                image_paths.append(image_path)
                labels.append(class_idx)
        class_indices = [i for i, label in enumerate(labels) if label == class_idx]
        num_samples = len(class_indices)
        if num_samples > biggest_class_size:
            biggest_class_size = num_samples

    target_samples_per_class = int(biggest_class_size * 1.25)
    for class_idx, class_name in tqdm(enumerate(class_names)):
        class_indices = [i for i, label in enumerate(labels) if label == class_idx]
        num_samples = len(class_indices)

        if num_samples < target_samples_per_class:
            # Augment the class to reach the target number of samples
            images_to_augment = target_samples_per_class - num_samples
            original_image_paths = [image_paths[i] for i in class_indices]

            for i in range(images_to_augment):
                # Randomly select an image from the class to augment
                random_image_path = random.choice(original_image_paths)
                image = cv2.imread(random_image_path)
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB

                # Apply augmentations
                augmented = augmentation_pipeline(image=image)
                augmented_image = augmented["image"]

                # Save the augmented image (optional, adjust filename as needed)
                augmented_image_name = f"{i}_augmented_{len(class_indices)}.png"
                augmented_image_path = os.path.join(
                    dataset_path, class_name, augmented_image_name
                )
                cv2.imwrite(
                    augmented_image_path,
                    cv2.cvtColor(augmented_image, cv2.COLOR_RGB2BGR),
                )  # Save as BGR


augmentation_pipeline = Compose(
    [
        HorizontalFlip(p=0.5),
        VerticalFlip(p=0.5),
        RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        HueSaturationValue(
            hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5
        ),
        GaussianBlur(blur_limit=(3, 7), p=0.3),
    ]
)

base_dir = "data/prep/"

for config in os.listdir(base_dir):
    for split in os.listdir(os.path.join(base_dir, config)):
        split_dir = os.path.join(base_dir, config, split)
        augment_and_balance_dataset(split_dir, augmentation_pipeline)